# 03 — Preprocessing: Penjelasan & Validasi Pipeline Pembersihan
**DA Gelar Rasa 2026 · Dashboard Kualitas Udara & Kesehatan**

Notebook ini **tidak membersihkan data**; pembersihan dijalankan oleh `etl/run_preprocessing.py` (memanggil fungsi di `src/clean.py`).
Di sini kita membaca keluarannya untuk (1) menjelaskan setiap langkah beserta dampaknya, (2) memvalidasi hasil secara independen, dan
(3) membuktikan keputusan yang diambil untuk stasiun bermasalah (terutama **JMB-02** dan **MER-01**).

| Langkah | Fungsi (`src/clean.py`) | Aturan ringkas |
|---|---|---|
| 1 | `bersihkan_stasiun` | Teks galat (`0xDEADBEEF`, `ERR_*`, `#VALUE!`, `INF`…) → `NaN`, cast ke numerik |
| 2 | `bersihkan_stasiun` | `*_ppm` → µg/m³: PM ×1000; gas ppm × BM / 24,45 × 1000. Divalidasi dengan median stasiun sejenis |
| 3 | `bersihkan_stasiun` | Parse timestamp campuran; bulatkan ke jam terdekat (bacaan sub-jam: floor); buang yang di luar periode |
| 4 | `bersihkan_stasiun` | Sentinel (±999, ±9999, -9999), polutan negatif, cuaca di luar batas fisik → `NaN`. Puncak PM >500 yang asli **dipertahankan** |
| 5 | `bersihkan_stasiun` | Hanya `qc_flag = VALID`; beberapa baris pada jam yang sama dirata-rata |
| 6 | `agregasi_harian` | Rata-rata harian hanya bila ≥ 18 dari 24 jam valid (75%) |
| 7 | `fitur_jam`, `fitur_harian`, `tandai_episode` | ISPU, flag baku mutu, kalender, musim, pulau, kelompok lokasi, episode kabut asap |
| 8 | `gabung_kesehatan`, `bangun_sites_clean` | sites + sensors + hospitals + kesehatan; PM2.5 lag 0–7 dan rata-rata 3 hari |

Cara menjalankan ulang pipeline: `python etl/run_preprocessing.py` (± 30 detik). Skrip berhenti tanpa menulis berkas bila ada validasi yang gagal.

In [1]:
import sys, warnings
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.io as pio
from IPython.display import display, Markdown

from src import config as cfg, style, profiling as P, clean as C, plots_preprocessing as G

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 140); pd.set_option("display.width", 220)
pio.renderers.default = "plotly_mimetype+notebook_connected"
cfg.FIG_DIR.mkdir(parents=True, exist_ok=True); cfg.FINDINGS_DIR.mkdir(parents=True, exist_ok=True)

PR = cfg.PROC_DIR
wajib = ["aq_hourly", "aq_daily", "health_aq_daily", "sites_clean", "qc_summary", "preprocessing_stat_pm25", "preprocessing_validasi", "preprocessing_validasi_satuan"]
hilang = [n for n in wajib if not (PR / f"{n}.parquet").exists()] + ([] if (cfg.REPORTS_DIR / "preprocessing_log.csv").exists() else ["preprocessing_log.csv"])
assert not hilang, f"Jalankan dulu: python etl/run_preprocessing.py  (belum ada: {hilang})"

hourly = pd.read_parquet(PR / "aq_hourly.parquet"); daily = pd.read_parquet(PR / "aq_daily.parquet"); health = pd.read_parquet(PR / "health_aq_daily.parquet")
sites = pd.read_parquet(PR / "sites_clean.parquet"); qc = pd.read_parquet(PR / "qc_summary.parquet")
stat = pd.read_parquet(PR / "preprocessing_stat_pm25.parquet"); vtbl = pd.read_parquet(PR / "preprocessing_validasi.parquet"); vsat = pd.read_parquet(PR / "preprocessing_validasi_satuan.parquet")
log = pd.read_csv(cfg.REPORTS_DIR / "preprocessing_log.csv", encoding="utf-8-sig")
prof02 = pd.read_parquet(PR / "profiling" / "profil_stasiun.parquet").set_index("kode")     # keluaran notebook 02 (data mentah)
urutan = G.urutan_stasiun(sites)

rp = lambda x: f"{int(round(x)):,}".replace(",", ".")
def fl(x, n=1):
    """Angka desimal gaya Indonesia: pemisah ribuan titik, desimal koma."""
    return "–" if pd.isna(x) else f"{x:,.{n}f}".replace(",", "X").replace(".", ",").replace("X", ".")
def md_t(df, desimal=1):
    """Tabel markdown dengan angka gaya Indonesia (tanpa penguraian angka otomatis oleh tabulate)."""
    x = df.copy()
    for c in x.columns:
        if pd.api.types.is_float_dtype(x[c]):
            x[c] = x[c].map(lambda v: fl(v, desimal))
    return x.to_markdown(index=False, disable_numparse=True)
def tandai_positif(df, kolom):
    return df.style.map(lambda v: "background-color:#FDE9C8" if isinstance(v, (int, float, np.integer, np.floating)) and v > 0 else "", subset=kolom)
print("Tabel dimuat. Stasiun:", len(sites), "| log:", rp(len(log)), "entri")

Tabel dimuat. Stasiun: 27 | log: 883 entri


## 1. Keluaran pipeline

In [2]:
grain = {"aq_hourly": "1 baris = stasiun × jam (hanya jam dengan ≥1 baris VALID)", "aq_daily": "1 baris = stasiun × hari (grid kalender lengkap)",
         "health_aq_daily": "1 baris = RS × hari + paparan PM2.5 stasiun pasangan", "sites_clean": "1 baris = stasiun (sites + sensors + hospitals)",
         "qc_summary": "1 baris = stasiun (kualitas & status analisis)"}
info = []
for nama, df in [("aq_hourly", hourly), ("aq_daily", daily), ("health_aq_daily", health), ("sites_clean", sites), ("qc_summary", qc)]:
    info.append(dict(tabel=nama, baris=len(df), kolom=df.shape[1], ukuran_mb=round((PR / f"{nama}.parquet").stat().st_size / 1e6, 2), grain=grain[nama]))
display(pd.DataFrame(info))
print("Stasiun di aq_hourly:", hourly.kode_stasiun.nunique(), "| di aq_daily:", daily.kode_stasiun.nunique(), "| dikecualikan:", sorted(set(sites.kode_stasiun) - set(hourly.kode_stasiun.astype(str))))
for nama, df in [("aq_hourly", hourly), ("aq_daily", daily), ("health_aq_daily", health)]:
    print(f"\n{nama} — contoh 2 baris (transpos):")
    display(df.head(2).T.rename(columns={0: "baris_1", 1: "baris_2"}) if len(df) > 1 else df.head(1).T)

,tabel,baris,kolom,ukuran_mb,grain
0,aq_hourly,591047,24,10.63,1 baris = stasiun × jam (hanya jam dengan ≥1 baris VALID)
1,aq_daily,26468,34,1.30,1 baris = stasiun × hari (grid kalender lengkap)
2,health_aq_daily,27486,47,1.67,1 baris = RS × hari + paparan PM2.5 stasiun pasangan
3,sites_clean,27,34,0.03,1 baris = stasiun (sites + sensors + hospitals)
4,qc_summary,27,35,0.03,1 baris = stasiun (kualitas & status analisis)


Stasiun di aq_hourly: 26 | di aq_daily: 26 | dikecualikan: ['JMB-02']

aq_hourly — contoh 2 baris (transpos):


,baris_1,baris_2
kode_stasiun,BGL-01,BGL-01
jam,2024-01-01 00:00:00,2024-01-01 01:00:00
pm25_ug_m3,20.6,20.6
pm10_ug_m3,29.8,32.0
so2_ug_m3,7.7,7.4
co_ug_m3,363.4,335.8
o3_ug_m3,11.4,11.8
no2_ug_m3,19.7,19.3
temperature_c,24.7,24.3
humidity_pct,90.8,89.9



aq_daily — contoh 2 baris (transpos):


,baris_1,baris_2
kode_stasiun,BGL-01,BGL-01
tanggal,2024-01-01 00:00:00,2024-01-02 00:00:00
pm25_ug_m3,20.1125,20.83913
pm10_ug_m3,29.9,30.778261
so2_ug_m3,7.1875,7.095652
co_ug_m3,355.8,343.282609
o3_ug_m3,13.75,12.991304
no2_ug_m3,20.204167,19.530435
temperature_c,25.866667,25.86087
humidity_pct,78.6875,78.008696



health_aq_daily — contoh 2 baris (transpos):


,baris_1,baris_2
date,2024-01-01 00:00:00,2024-01-02 00:00:00
urutan_hari_rs,1,2
hospital_id,RS-XXX-01,RS-XXX-01
dow,Monday,Tuesday
temp_c,28.4,28.7
dptp_c,23.8,23.9
rhum_pct,76.1,75.0
resp_admissions_under5,12,13
resp_admissions_over60,25,22
asthma_emergency_cases,12,6


## 2. Log preprocessing
`reports/preprocessing_log.csv`: satu baris per (langkah, entitas, metrik) dengan jumlah terdampak, satuan, alasan, dan — bila baris berubah — jumlah **sebelum** dan **sesudah**.

In [3]:
log["metrik_dasar"] = log.metrik.str.replace(r"\[.*\]", "", regex=True)
stas = set(sites.kode_stasiun)
ls = log[log.entitas.isin(stas)]
ring = (ls.groupby(["langkah", "nama_langkah", "metrik_dasar", "satuan"], sort=False)
        .agg(jumlah_total=("jumlah", "sum"), stasiun_terdampak=("jumlah", lambda s: int((s > 0).sum()))).reset_index())
print("Ringkasan seluruh stasiun, per langkah dan metrik:")
display(ring)

Ringkasan seluruh stasiun, per langkah dan metrik:


,langkah,nama_langkah,metrik_dasar,satuan,jumlah_total,stasiun_terdampak
0,1,Teks galat → NaN & cast numerik,sel_teks_galat_menjadi_nan,sel,43171.0,1
1,1,Teks galat → NaN & cast numerik,sel_kosong_dibaca_nan,sel,193366.0,5
2,1,Teks galat → NaN & cast numerik,kolom_di_cast_numerik,kolom,270.0,27
3,2,Standarisasi satuan ke µg/m³,sel_dikonversi_ppm_ke_ugm3,sel,414240.0,3
4,3,"Parse timestamp, bulatkan ke jam, buang di luar periode",baris_timestamp_tak_terbaca,baris,0.0,0
5,3,"Parse timestamp, bulatkan ke jam, buang di luar periode",baris_di_luar_periode,baris,132.0,1
6,3,"Parse timestamp, bulatkan ke jam, buang di luar periode",baris_dibulatkan_ke_jam_terdekat,baris,411488.0,17
7,3,"Parse timestamp, bulatkan ke jam, buang di luar periode",baris_subjam_dipotong_ke_jam_penuh,baris,16025.0,2
8,3,"Parse timestamp, bulatkan ke jam, buang di luar periode",jumlah_baris,baris,132.0,1
9,4,Buang nilai mustahil & sentinel,sel_sentinel_menjadi_nan,sel,43185.0,2


In [4]:
print("Rincian label qc_flag yang dibuang pada langkah 5 (seluruh stasiun):")
b = ls[ls.metrik.str.startswith("baris_non_valid_dibuang[")].assign(label=lambda d: d.metrik.str.extract(r"\[(.*)\]")[0])
display(b.pivot_table(index="entitas", columns="label", values="jumlah", aggfunc="sum", fill_value=0).assign(Total=lambda d: d.sum(axis=1)))

print("\nJumlah entri log per langkah:")
display(log.groupby(["langkah", "nama_langkah"]).size().rename("entri").to_frame())

Rincian label qc_flag yang dibuang pada langkah 5 (seluruh stasiun):


label,CALIBRATING,COMM_LOST,CORRUPT,FATAL_ERR,GARBAGE,MAINTENANCE,MISSING,PARSER_FAIL,SENSOR_DEAD,SUSPECT,WARMUP,Total
entitas,,,,,,,,,,,,
BGL-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,125.0,0.0,125.0
BGR-01,256.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,488.0,0.0,744.0
BGR-02,253.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,484.0,0.0,737.0
BTM-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,154.0,0.0,154.0
IDP-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,110.0,0.0,110.0
JKT-01,367.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,611.0,0.0,978.0
JKT-02,343.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,609.0,0.0,952.0
JKT-03,366.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,596.0,0.0,962.0
JMB-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,168.0,0.0,168.0



Jumlah entri log per langkah:


,,entri
langkah,nama_langkah,
1,Teks galat → NaN & cast numerik,81
2,Standarisasi satuan ke µg/m³,27
3,"Parse timestamp, bulatkan ke jam, buang di luar periode",135
4,Buang nilai mustahil & sentinel,162
5,Filter qc_flag = VALID & agregasi per jam (rata-rata),160
6,Agregasi harian (≥ 18 dari 24 jam valid),130
7,Feature engineering,104
8,Penggabungan master + kesehatan,84


In [5]:
metrik_kunci = ["sel_teks_galat_menjadi_nan", "sel_dikonversi_ppm_ke_ugm3", "baris_timestamp_tak_terbaca", "baris_di_luar_periode",
                "baris_dibulatkan_ke_jam_terdekat", "baris_subjam_dipotong_ke_jam_penuh", "sel_sentinel_menjadi_nan", "sel_negatif_menjadi_nan",
                "sel_di_luar_batas_fisik_cuaca_menjadi_nan", "jumlah_baris_non_valid", "jumlah_baris_per_jam", "jam_valid_akhir",
                "hari_pm25_ditolak_kurang_dari_18_jam", "hari_pm25_valid", "episode_kabut_asap"]
pv = ls[ls.metrik.isin(metrik_kunci)].pivot_table(index="entitas", columns="metrik", values="jumlah", aggfunc="sum").reindex(index=urutan, columns=metrik_kunci)
pv.columns = [c.replace("_menjadi_nan", "").replace("sel_", "").replace("baris_", "").replace("jumlah_", "") for c in pv.columns]
print("Dampak per stasiun (sel = nilai; baris/jam/hari sesuai nama kolom). Sel berwarna = ada yang terdampak:")
display(tandai_positif(pv.fillna(0).astype(int), list(pv.columns)))

Dampak per stasiun (sel = nilai; baris/jam/hari sesuai nama kolom). Sel berwarna = ada yang terdampak:


,teks_galat,dikonversi_ppm_ke_ugm3,timestamp_tak_terbaca,di_luar_periode,dibulatkan_ke_jam_terdekat,subjam_dipotong_ke_jam_penuh,sentinel,negatif,di_luar_batas_fisik_cuaca,non_valid,per_jam,jam_valid_akhir,hari_pm25_ditolak_kurang_dari_18_jam,hari_pm25_valid,episode_kabut_asap
entitas,,,,,,,,,,,,,,,
BGL-01,0,0,0,0,24432,0,0,0,0,125,0,24307,0,1018,0
MDN-01,0,121056,0,0,20800,0,0,0,0,4160,0,16640,628,371,0
PSW-01,0,146592,0,0,24432,0,0,0,0,138,0,24294,0,1018,0
BTM-01,0,0,0,0,24432,0,0,0,0,154,0,24278,0,1018,0
JMB-01,0,0,0,0,24432,0,0,0,0,168,0,24264,0,1018,0
JMB-02,43171,0,0,132,24208,92,41107,33270,58910,24179,0,121,0,0,0
PKU-01,0,0,0,0,24432,0,0,0,0,132,0,24300,0,1018,0
PLB-01,0,0,0,0,0,15933,0,0,0,8934,8016,19431,322,696,10
IDP-01,0,0,0,0,0,0,0,0,0,110,0,24322,0,1018,0


In [6]:
print("Rekonsiliasi independen: baris mentah = tak terbaca + luar periode + non-VALID + VALID, dan VALID − digabung = jam akhir")
def L(k, m, kol="jumlah"):
    x = log[(log.entitas == k) & (log.metrik == m)]
    return int(x[kol].iloc[0]) if len(x) else 0
rek = []
for k in qc.kode_stasiun:
    n0 = int(qc.set_index("kode_stasiun").loc[k, "n_baris_mentah"])
    nv = L(k, "jumlah_baris_non_valid"); v = L(k, "jumlah_baris_non_valid", "sesudah")
    jumlah = L(k, "baris_timestamp_tak_terbaca") + L(k, "baris_di_luar_periode") + nv + v
    jam = L(k, "jam_valid_akhir", "sesudah")
    rek.append(dict(kode_stasiun=k, baris_mentah=n0, rekonstruksi=jumlah, cocok_baris=n0 == jumlah, valid=v, digabung=L(k, "jumlah_baris_per_jam"), jam_akhir=jam,
                    cocok_jam=(v - L(k, "jumlah_baris_per_jam")) == jam))
rek = pd.DataFrame(rek)
display(rek[~(rek.cocok_baris & rek.cocok_jam)] if (~(rek.cocok_baris & rek.cocok_jam)).any() else pd.DataFrame({"hasil": [f"Seluruh {len(rek)} stasiun cocok pada kedua rekonsiliasi"]}))
assert rek.cocok_baris.all() and rek.cocok_jam.all()

Rekonsiliasi independen: baris mentah = tak terbaca + luar periode + non-VALID + VALID, dan VALID − digabung = jam akhir


,hasil
0,Seluruh 27 stasiun cocok pada kedua rekonsiliasi


## 3. Validasi (assert)
Pemeriksaan dijalankan **dua kali**: oleh pipeline (jika gagal, berkas tidak ditulis) dan ulang di sini terhadap berkas yang sudah tersimpan.

In [7]:
ulang = C.validasi_output(hourly, daily, health, sites, qc, lanjut_jika_gagal=True)
display(ulang.style.map(lambda v: "background-color:#D6F0E0" if v is True else ("background-color:#F8D0CC" if v is False else ""), subset=["lulus"]))
assert ulang.lulus.all(), "Ada validasi yang gagal"
assert ulang.lulus.equals(vtbl.lulus), "Hasil validasi tersimpan berbeda dari validasi ulang"
print(f"{int(ulang.lulus.sum())}/{len(ulang)} pemeriksaan lulus")

,pemeriksaan,lulus,detail
0,"Tidak ada duplikat (stasiun, jam) pada aq_hourly",True,0 duplikat
1,"Tidak ada duplikat (stasiun, tanggal) pada aq_daily",True,
2,"Tidak ada duplikat (hospital_id, tanggal) pada health_aq_daily",True,
3,Semua jam berada di dalam periode 2024-01-01…2026-10-14 23:00,True,
4,Semua jam tepat di awal jam (menit=detik=0),True,
5,"Semua polutan (PM, SO2, CO, O3, NO2) ≥ 0 di aq_hourly",True,"{'pm25_ug_m3': 0, 'pm10_ug_m3': 0, 'so2_ug_m3': 0, 'co_ug_m3': 0, 'o3_ug_m3': 0, 'no2_ug_m3': 0}"
6,Semua polutan ≥ 0 di aq_daily,True,"{'pm25_ug_m3': 0, 'pm10_ug_m3': 0, 'so2_ug_m3': 0, 'co_ug_m3': 0, 'o3_ug_m3': 0, 'no2_ug_m3': 0}"
7,"Satuan seragam: semua kolom polutan berakhiran _ug_m3, tidak ada _ppm",True,
8,Cuaca dalam batas fisik,True,
9,Tidak ada nilai sentinel tersisa (selain CO=999 yang wajar),True,


21/21 pemeriksaan lulus


## 4. Validasi konversi satuan ppm → µg/m³
Median stasiun berunit ppm (**setelah** konversi, baris VALID) dibandingkan dengan median stasiun **sejenis** (kelompok lokasi sama, berunit µg/m³, bukan Dikecualikan).
PM2.5 juga dibandingkan dengan `baseline_pm25`. Kolom `rasio_tanpa_konversi` memperlihatkan ke mana skala akan jatuh bila konversi tidak dilakukan.

In [8]:
v = vsat.copy()
print("Stasiun sejenis:", v.groupby("kode_stasiun").kelompok_lokasi.first().to_dict())
display(v.round(3).style.map(lambda x: "background-color:#D6F0E0" if isinstance(x, float) and 0.25 <= x <= 4 else "", subset=["rasio"]))
print(f"Rasio setelah konversi : {fl(v.rasio.min(), 2)} – {fl(v.rasio.max(), 2)}  (1,00 = sama dengan stasiun sejenis)")
print(f"Rasio tanpa konversi   : {fl(v.rasio_tanpa_konversi.min(), 5)} – {fl(v.rasio_tanpa_konversi.max(), 5)}  (jauh dari 1 → konversi diperlukan)")
print(f"PM2.5 vs baseline_pm25 : {fl(v.rasio_vs_baseline.min(), 2)} – {fl(v.rasio_vs_baseline.max(), 2)}")

Stasiun sejenis: {'BJB-01': 'Gambut', 'MDN-01': 'Perkotaan', 'PSW-01': 'Perkotaan'}


,kode_stasiun,kelompok_lokasi,polutan,n_stasiun_sejenis,median_setelah_konversi,median_stasiun_sejenis,rasio,rasio_tanpa_konversi,baseline_pm25,rasio_vs_baseline
0,BJB-01,Gambut,pm25,6,45.100000,28.585000,1.578000,0.002000,35.000000,1.289000
1,BJB-01,Gambut,pm10,6,65.300000,42.250000,1.546000,0.002000,nan,nan
2,BJB-01,Gambut,so2,6,8.385000,7.200000,1.165000,0.000000,nan,nan
3,BJB-01,Gambut,co,6,544.162000,350.300000,1.553000,0.001000,nan,nan
4,BJB-01,Gambut,o3,6,17.669000,10.450000,1.691000,0.001000,nan,nan
5,BJB-01,Gambut,no2,6,30.862000,18.100000,1.705000,0.001000,nan,nan
6,MDN-01,Perkotaan,pm25,5,46.500000,30.100000,1.545000,0.002000,42.800000,1.086000
7,MDN-01,Perkotaan,pm10,5,69.300000,44.900000,1.543000,0.002000,nan,nan
8,MDN-01,Perkotaan,so2,5,9.172000,10.200000,0.899000,0.000000,nan,nan
9,MDN-01,Perkotaan,co,5,654.139000,497.000000,1.316000,0.001000,nan,nan


Rasio setelah konversi : 0,65 – 2,04  (1,00 = sama dengan stasiun sejenis)
Rasio tanpa konversi   : 0,00028 – 0,00158  (jauh dari 1 → konversi diperlukan)
PM2.5 vs baseline_pm25 : 0,80 – 1,29


## 5. Keputusan untuk stasiun bermasalah
**Status analisis** ditetapkan otomatis dari cakupan PM2.5 (`qc_summary`): *Dikecualikan* bila jam valid < 10% (tidak masuk `aq_hourly`/`aq_daily`);
*Layak* bila cakupan jam, hari, dan hari musim kemarau semuanya ≥ 75%; selain itu *Terbatas* (tetap ada, dengan peringatan).

In [9]:
kol = ["kode_stasiun", "status_sensor", "persen_valid_mentah", "persen_jam_pm25_valid", "persen_hari_pm25_valid", "persen_hari_pm25_valid_kemarau",
       "cakupan_terendah_persen", "gap_terpanjang_jam", "status_analisis"]
display(qc[kol].sort_values("cakupan_terendah_persen").head(8).reset_index(drop=True))
print("Jumlah stasiun per status:", qc.status_analisis.value_counts().to_dict())
for k in ["JMB-02", "MER-01", "MDN-01", "PLB-01", "SM2-01", "BJB-01"]:
    print(f"\n[{k}] {qc.set_index('kode_stasiun').loc[k, 'catatan']}")

,kode_stasiun,status_sensor,persen_valid_mentah,persen_jam_pm25_valid,persen_hari_pm25_valid,persen_hari_pm25_valid_kemarau,cakupan_terendah_persen,gap_terpanjang_jam,status_analisis
0,MER-01,Intermittent,20.02,20.02,0.20,0.37,0.20,507,Terbatas
1,JMB-02,Defective,0.50,0.50,NaN,NaN,0.50,19173,Dikecualikan
2,MDN-01,Intermittent,80.00,68.11,36.44,37.20,36.44,122,Terbatas
3,SM2-01,Active,85.00,85.00,75.74,53.83,53.83,14,Terbatas
4,PLB-01,Intermittent,75.44,79.53,68.37,72.34,68.37,13,Terbatas
5,JKT-01,Active,96.00,96.00,100.00,100.00,96.00,2,Layak
6,JKT-03,Active,96.06,96.06,100.00,100.00,96.06,3,Layak
7,JKT-02,Active,96.10,96.10,100.00,100.00,96.10,3,Layak


Jumlah stasiun per status: {'Layak': 22, 'Terbatas': 4, 'Dikecualikan': 1}

[JMB-02] Dikecualikan dari aq_hourly, aq_daily, dan paparan kesehatan. Sensor Defective; hanya 0,5% baris VALID, tersebar acak dan berhenti setelah Agustus 2024; 99,5% baris berisi galat/mustahil. Dipertahankan di sites_clean & qc_summary sebagai bukti kebutuhan jaringan (Tujuan 5).

[MER-01] Terbatas. ±20% jam VALID tersebar acak (median 6 jam VALID per hari), sehingga hampir tidak ada hari yang memenuhi ≥18 jam. Dipakai untuk profil per jam/klimatologi (nilai VALID wajar), TIDAK untuk model harian/lag kesehatan. Aturan 75% tidak dilonggarkan.

[MDN-01] Terbatas. 3.632 jam tidak tercatat dan 20% baris non-VALID (WARMUP/MAINTENANCE/CALIBRATING); satuan ppm dikonversi.

[PLB-01] Sensor Intermittent: 5.136 baris MISSING membuat cakupan harian jauh lebih rendah daripada cakupan jam. Logger merekam tiap 15 menit pada Sep–Okt (dirata-rata per jam).

[SM2-01] Nilai saturasi 999 (SUSPECT) pada Jul–Okt dibuang, sehingg

### 5.1 JMB-02 (Muaro Jambi): **dikecualikan** dari tabel analisis
Bukti dari data mentah (profil notebook 02) dan dari baris VALID yang tersisa:

In [10]:
j = prof02.loc["JMB-02"]
print(f"Baris VALID mentah        : {int(j.n_valid)} dari {rp(j.n_baris)} ({fl(j.persen_valid, 2)}%)")
print(f"Baris berisi teks galat   : {rp(j.n_baris_teks)} ({fl(100 * j.n_baris_teks / j.n_baris, 1)}%)")
print(f"Baris berisi nilai mustahil: {rp(j.n_baris_mustahil)} ({fl(100 * j.n_baris_mustahil / j.n_baris, 1)}%)")
print(f"Baris bertimestamp 1970    : {int(j.n_ts_luar_jendela)}")
raw = P.baca_mentah(cfg.RAW_DIR / "JMB-02_MuaroJambi_AQMS.csv")
ts = pd.to_datetime(raw.timestamp_start, format="mixed", errors="coerce")
vj = raw[raw.qc_flag.eq("VALID") & (ts >= cfg.WINDOW_START)]
tj = ts[vj.index].dt.to_period("M").value_counts().sort_index()
print(f"\nBaris VALID per bulan (periode 2024+): terakhir pada {tj.index.max()}; tidak ada satu pun baris VALID setelahnya selama {(cfg.WINDOW_END.to_period('M') - tj.index.max()).n} bulan")
display(tj.rename("baris_valid").to_frame().T)
hj = ts[vj.index].dt.floor("D").value_counts()
print(f"Hari yang memenuhi ≥18 jam VALID: {(hj >= 18).sum()} dari {len(hj)} hari yang punya ≥1 baris VALID")
r = health[health.kode_stasiun == "JMB-02"]
print(f"\nDampak ke kesehatan: RS-XXX-27 tetap ada di health_aq_daily ({len(r)} hari) dengan paparan kosong (ada_paparan_pm25 = {int(r.ada_paparan_pm25.sum())}) dan status '{r.status_analisis_stasiun.iloc[0]}'")

Baris VALID mentah        : 122 dari 24.432 (0,50%)
Baris berisi teks galat   : 8.997 (36,8%)
Baris berisi nilai mustahil: 24.308 (99,5%)
Baris bertimestamp 1970    : 132



Baris VALID per bulan (periode 2024+): terakhir pada 2024-08; tidak ada satu pun baris VALID setelahnya selama 26 bulan


timestamp_start,2024-01,2024-02,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08
baris_valid,25,18,14,22,8,14,16,4


Hari yang memenuhi ≥18 jam VALID: 0 dari 98 hari yang punya ≥1 baris VALID

Dampak ke kesehatan: RS-XXX-27 tetap ada di health_aq_daily (1018 hari) dengan paparan kosong (ada_paparan_pm25 = 0) dan status 'Dikecualikan'


**Keputusan JMB-02.** Dikecualikan dari `aq_hourly`, `aq_daily`, dan paparan kesehatan. Alasannya bukan hanya jumlah VALID yang kecil, melainkan sensor berhenti menghasilkan baris VALID setelah pertengahan 2024
(*Defective*), sehingga tidak ada cara menyusun deret harian yang sah. Tidak diimputasi dan tidak dilonggarkan. Stasiun ini **tetap ada** di `sites_clean` dan `qc_summary` sebagai bukti kebutuhan perbaikan jaringan (Tujuan 5).
RS pasangannya (RS-XXX-27) dipertahankan untuk deskripsi kasus tanpa paparan.

### 5.2 MER-01 (Merauke): **Terbatas** — dipakai untuk profil per jam, tidak untuk model harian

In [11]:
m = hourly[hourly.kode_stasiun == "MER-01"].copy()
m["tgl"] = m.jam.dt.floor("D")
jph = m.groupby("tgl").size()
print(f"Jam PM2.5 valid: {rp(len(m))} dari {rp(cfg.EXPECTED_HOURS)} ({fl(100 * len(m) / cfg.EXPECTED_HOURS, 1)}%); hari yang punya ≥1 jam: {len(jph)} dari 1.018; median {fl(jph.median(), 0)} jam/hari")
ambang = pd.DataFrame({"ambang_jam_per_hari": [18, 12, 6, 1]})
ambang["hari_memenuhi"] = [(jph >= a).sum() for a in ambang.ambang_jam_per_hari]
ambang["persen_dari_1018_hari"] = (100 * ambang.hari_memenuhi / 1018).round(2)
print("\nJika ambang harian diubah (hanya untuk menunjukkan mengapa aturan 75% tidak dilonggarkan):")
display(ambang)
jb = m.groupby(m.jam.dt.to_period("M")).size()
print(f"Jam valid per bulan: min {jb.min()}, median {fl(jb.median(), 0)}, maks {jb.max()} → tersebar merata, bukan terkumpul pada satu musim")
blok = (m.jam.sort_values().diff() != pd.Timedelta(hours=1)).cumsum().nunique()
print(f"Jam valid terbagi dalam {blok} blok berurutan (rata-rata {fl(len(m) / blok, 1)} jam per blok)")
pe = hourly[hourly.kode_stasiun.isin(["MER-01", "SRG-01"])].groupby("kode_stasiun", observed=True)[["pm25_ug_m3", "pm10_ug_m3", "temperature_c", "humidity_pct"]].agg(["median", "max"]).round(1)
print("\nKewajaran nilai VALID MER-01 dibanding SRG-01 (stasiun pesisir/latar di Papua):")
display(pe)

Jam PM2.5 valid: 4.891 dari 24.432 (20,0%); hari yang punya ≥1 jam: 801 dari 1.018; median 6 jam/hari

Jika ambang harian diubah (hanya untuk menunjukkan mengapa aturan 75% tidak dilonggarkan):


,ambang_jam_per_hari,hari_memenuhi,persen_dari_1018_hari
0,18,2,0.20
1,12,54,5.30
2,6,430,42.24
3,1,801,78.68


Jam valid per bulan: min 66, median 148, maks 213 → tersebar merata, bukan terkumpul pada satu musim
Jam valid terbagi dalam 1837 blok berurutan (rata-rata 2,7 jam per blok)

Kewajaran nilai VALID MER-01 dibanding SRG-01 (stasiun pesisir/latar di Papua):


pm25_ug_m3       pm10_ug_m3       temperature_c       humidity_pct      
                 median   max     median   max        median   max       median   max
kode_stasiun                                                                         
MER-01             16.0  29.2       24.7  48.8          28.0  34.9         80.1  98.0
SRG-01             17.9  58.6       26.8  93.8          24.5  30.5         85.0  92.6

**Keputusan MER-01.** Hanya ±20% jam yang VALID, tersebar acak (ratusan blok pendek), sehingga **hampir tidak ada hari** yang memenuhi ≥ 18 jam. Nilai VALID-nya wajar (setara stasiun latar) sehingga data per jam **dipertahankan** untuk profil diurnal/klimatologi
dan sebagai bukti kualitas jaringan di Papua Selatan. Namun:
- ia **tidak dipakai** untuk model harian / efek jeda kesehatan (paparan RS-XXX-23 praktis kosong);
- aturan 75% **tidak dilonggarkan**, karena rata-rata dari 6 jam tidak mewakili satu hari;
- karena nilai VALID tersebar merata antar-bulan, tidak ada bukti bias musiman, tetapi puncak kabut asap (bila ada) tidak mungkin tertangkap.

### 5.3 SM2-01 (Samarinda): sensor saturasi pada musim asap → hari berasap banyak hilang
Aturan "hanya VALID" membuang nilai saturasi 999 yang bertanda `SUSPECT`. Karena itu hari-hari pada musim kemarau lebih sering gagal aturan 18 jam.
Kolom `pm25_ug_m3_longgar` (≥12 jam) disediakan **hanya untuk analisis sensitivitas**.

In [12]:
fig = G.fig_cakupan_harian_bulan(daily, "SM2-01")
style.simpan_html(fig, "03_sm2_cakupan_harian_bulan"); fig.show()

sat = P.bulan_nilai_sentinel("SM2-01", 999.0)
nama = {1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "Mei", 6: "Jun", 7: "Jul", 8: "Agu", 9: "Sep", 10: "Okt", 11: "Nov", 12: "Des"}
print("Baris bernilai 999 (saturasi, SUSPECT) yang dibuang, per bulan kalender:", {nama[k]: int(v) for k, v in sat.items()})

s = daily[daily.kode_stasiun == "SM2-01"].copy()
kem_hr = s[s.bulan.isin([7, 8, 9, 10])]
print(f"\nJul–Okt: hari sah (≥18 jam) {fl(100 * kem_hr.pm25_ug_m3.notna().mean(), 1)}% vs {fl(100 * kem_hr.pm25_ug_m3_longgar.notna().mean(), 1)}% (≥12 jam); bulan lain: {fl(100 * s[~s.bulan.isin([7, 8, 9, 10])].pm25_ug_m3.notna().mean(), 1)}%")

def episode_ringkas(d, kolom):
    x = d[["kode_stasiun", "tanggal", kolom]].rename(columns={kolom: "pm25_ug_m3"}).copy()
    x["kode_stasiun"] = x.kode_stasiun.astype(str)
    e = C.tandai_episode(x, C.LogPreprocessing())
    return dict(episode=int(e.id_episode.nunique()), hari_episode=int(e.episode_asap.sum()), hari_pm25_diatas_150=int((x.pm25_ug_m3 > 150).sum()))
bandingan = pd.DataFrame({"aturan ≥18 jam (dipakai)": episode_ringkas(s, "pm25_ug_m3"), "aturan ≥12 jam (sensitivitas)": episode_ringkas(s, "pm25_ug_m3_longgar")})
print("\nDeteksi episode kabut asap SM2-01 menurut aturan kelengkapan harian:")
display(bandingan)

Baris bernilai 999 (saturasi, SUSPECT) yang dibuang, per bulan kalender: {'Jul': 285, 'Agu': 267, 'Sep': 270, 'Okt': 217}

Jul–Okt: hari sah (≥18 jam) 29,8% vs 97,2% (≥12 jam); bulan lain: 100,0%

Deteksi episode kabut asap SM2-01 menurut aturan kelengkapan harian:


,aturan ≥18 jam (dipakai),aturan ≥12 jam (sensitivitas)
episode,6,28
hari_episode,20,223
hari_pm25_diatas_150,76,254


## 6. Grafik sebelum vs sesudah

In [13]:
fig = G.fig_distribusi_sebelum_sesudah(stat, urutan)
style.simpan_html(fig, "03_distribusi_pm25_sebelum_sesudah"); fig.show()

sb = stat[stat.tahap == "sebelum"].set_index("kode_stasiun"); sa = stat[stat.tahap == "sesudah"].set_index("kode_stasiun")
cmp = pd.DataFrame({"satuan_asli": sites.set_index("kode_stasiun").satuan_asli, "median_sebelum": sb["median"], "median_sesudah": sa["median"],
                    "maks_sebelum": sb["maks"], "maks_sesudah": sa["maks"], "negatif_sebelum": sb["n_negatif"]}).reindex(urutan)
print("Ringkasan numerik (median/maks PM2.5; sebelum = skala label file, sesudah = µg/m³):")
display(cmp.round(3))

Ringkasan numerik (median/maks PM2.5; sebelum = skala label file, sesudah = µg/m³):


,satuan_asli,median_sebelum,median_sesudah,maks_sebelum,maks_sesudah,negatif_sebelum
kode_stasiun,,,,,,
BGL-01,ug/m3,25.100,25.10,65.000,65.000,0
MDN-01,ppm,0.047,46.50,0.269,86.300,0
PSW-01,ppm,0.044,43.50,0.134,133.500,0
BTM-01,ug/m3,38.500,38.50,191.500,191.500,0
JMB-01,ug/m3,32.300,32.30,204.100,203.400,0
JMB-02,ug/m3,-278.885,NaN,99954.070,NaN,7589
PKU-01,ug/m3,35.300,35.30,201.300,201.300,0
PLB-01,ug/m3,72.800,45.40,285.100,282.675,0
IDP-01,ug/m3,8.900,8.90,24.500,24.500,0


In [14]:
raw_plb = P.baca_mentah(cfg.RAW_DIR / "PLB-01_TalangBetutu_AQMS.csv"); raw_bjb = P.baca_mentah(cfg.RAW_DIR / "BJB-01_Banjarbaru_AQMS.csv")
gam = sites[(sites.kelompok_lokasi == "Gambut") & (sites.kode_stasiun != "BJB-01") & (sites.status_analisis == "Layak")].kode_stasiun
dg = daily[daily.kode_stasiun.isin(gam)].groupby("tanggal").pm25_ug_m3.mean()
fig = G.fig_deret_plb_bjb(raw_plb, hourly[hourly.kode_stasiun == "PLB-01"], raw_bjb, daily[daily.kode_stasiun == "BJB-01"], dg)
style.simpan_html(fig, "03_deret_plb_bjb_sebelum_sesudah"); fig.show()
print("Stasiun gambut pembanding (status Layak, tanpa BJB-01):", ", ".join(gam))

Stasiun gambut pembanding (status Layak, tanpa BJB-01): JMB-01, KTG-02, PKN-01, PKU-01, STG-01


## 7. Feature engineering: validasi

In [15]:
print("Kolom fitur aq_hourly :", [c for c in hourly.columns if c not in ("kode_stasiun", "jam", "n_baca") + tuple(C.VAR_NUM)])
print("Kolom fitur aq_daily  :", [c for c in daily.columns if c not in ("kode_stasiun", "tanggal") + tuple(C.VAR_NUM)])

# 1) ISPU: breakpoint
uji = pd.Series([15.5, 15.6, 55.4, 55.5, 150.4, 150.5, 250.4, 250.5, 500.0, np.nan])
tabel = pd.DataFrame({"pm25": uji, "indeks_ispu": C.indeks_ispu_pm25(uji), "kategori": C.kategori_ispu_pm25(uji).astype(str)})
display(tabel.T)
assert C.indeks_ispu_pm25(pd.Series([15.5, 55.4, 150.4, 250.4, 500.0])).tolist() == [50, 100, 200, 300, 500]
# 2) konsistensi kolom turunan
d = daily
assert (d.kategori_ispu_pm25.astype(str).where(d.pm25_ug_m3.notna(), "nan") == C.kategori_ispu_pm25(d.pm25_ug_m3).astype(str).where(d.pm25_ug_m3.notna(), "nan")).all()
assert (d.lewat_bm_pm25.dropna() == (d.pm25_ug_m3[d.pm25_ug_m3.notna()] > cfg.BM_HARIAN["pm25"])).all()
assert (d.loc[d.episode_asap, "pm25_ug_m3"] > cfg.EPISODE_PM25_AMBANG).all()
panjang = d[d.episode_asap].groupby("id_episode").size()
assert (panjang >= cfg.EPISODE_MIN_HARI).all()
assert hourly.loc[hourly.bulan.isin([5, 6, 7, 8, 9, 10]), "musim"].eq("Kemarau").all() and hourly.loc[~hourly.bulan.isin([5, 6, 7, 8, 9, 10]), "musim"].eq("Hujan").all()
mer = hourly[hourly.kode_stasiun == "MER-01"]
assert ((mer.jam_wib + 2) % 24 == mer.jam_lokal).all()
print("Semua assert fitur lulus: ISPU, flag baku mutu, episode (≥3 hari, >150), musim kemarau Mei–Okt, jam_lokal WIT (+2)")

Kolom fitur aq_hourly : ['tahun', 'bulan', 'hari_ke', 'hari', 'akhir_pekan', 'musim', 'jam_wib', 'jam_lokal', 'pulau', 'kelompok_lokasi', 'pm_ekstrem']
Kolom fitur aq_daily  : ['pm25_ug_m3_longgar', 'pm25_maks_jam_ug_m3', 'n_jam_pm25', 'n_jam_pm10', 'n_jam_data', 'tahun', 'bulan', 'hari_ke', 'hari', 'akhir_pekan', 'musim', 'pulau', 'kelompok_lokasi', 'ispu_pm25', 'kategori_ispu_pm25', 'lewat_bm_pm25', 'lewat_who_pm25', 'lewat_bm_pm10', 'lewat_who_pm10', 'episode_asap', 'id_episode', 'hari_ke_episode']


,0,1,2,3,4,5,6,7,8,9
pm25,15.5,15.6,55.4,55.5,150.4,150.5,250.4,250.5,500.0,NaN
indeks_ispu,50.0,50.1,100.0,100.1,200.0,200.1,300.0,300.1,500.0,NaN
kategori,Baik,Sedang,Sedang,Tidak Sehat,Tidak Sehat,Sangat Tidak Sehat,Sangat Tidak Sehat,Berbahaya,Berbahaya,nan


Semua assert fitur lulus: ISPU, flag baku mutu, episode (≥3 hari, >150), musim kemarau Mei–Okt, jam_lokal WIT (+2)


In [16]:
print("Kategori ISPU PM2.5 harian (hari dengan rata-rata sah):")
kat = d.kategori_ispu_pm25.value_counts().reindex(cfg.ISPU_KATEGORI)
display(pd.DataFrame({"hari": kat, "persen": (100 * kat / kat.sum()).round(2)}).T)
fig = G.fig_ispu_per_pulau(d); style.simpan_html(fig, "03_ispu_per_pulau"); fig.show()

print("Episode kabut asap (PM2.5 harian > 150 µg/m³, ≥ 3 hari kalender berturut-turut):")
ep = (d[d.episode_asap].groupby(["kode_stasiun", "id_episode"], observed=True)
      .agg(awal=("tanggal", "min"), akhir=("tanggal", "max"), hari=("tanggal", "size"), pm25_maks_harian=("pm25_ug_m3", "max")).reset_index())
ep["kelompok_lokasi"] = ep.kode_stasiun.astype(str).map(sites.set_index("kode_stasiun").kelompok_lokasi)
display(ep.round(1))
print("Hari episode per bulan kalender:", d[d.episode_asap].bulan.value_counts().sort_index().to_dict())

Kategori ISPU PM2.5 harian (hari dengan rata-rata sah):


kategori_ispu_pm25,Baik,Sedang,Tidak Sehat,Sangat Tidak Sehat,Berbahaya
hari,4287.00,15782.00,3694.00,172.00,301.00
persen,17.69,65.12,15.24,0.71,1.24


Episode kabut asap (PM2.5 harian > 150 µg/m³, ≥ 3 hari kalender berturut-turut):


,kode_stasiun,id_episode,awal,akhir,hari,pm25_maks_harian,kelompok_lokasi
0,BJB-01,BJB-01-E01,2024-09-07,2025-03-29,204,336.7,Gambut
1,KTG-01,KTG-01-E01,2026-09-01,2026-10-14,44,1127.6,Perkotaan
2,KTG-02,KTG-02-E01,2026-09-01,2026-10-14,44,1119.6,Gambut
3,PKN-01,PKN-01-E01,2026-10-09,2026-10-14,6,628.0,Gambut
4,PLB-01,PLB-01-E01,2025-08-30,2025-09-05,7,177.7,Gambut
5,PLB-01,PLB-01-E02,2025-09-16,2025-09-18,3,177.6,Gambut
6,PLB-01,PLB-01-E03,2025-09-26,2025-10-05,10,177.3,Gambut
7,PLB-01,PLB-01-E04,2025-10-14,2025-10-18,5,174.1,Gambut
8,PLB-01,PLB-01-E05,2025-10-26,2025-10-30,5,174.1,Gambut
9,PLB-01,PLB-01-E06,2026-08-05,2026-08-08,4,181.9,Gambut


Hari episode per bulan kalender: {1: 31, 2: 28, 3: 29, 8: 11, 9: 132, 10: 104, 11: 30, 12: 31}


In [17]:
print("Kelulusan aturan 18 jam dan kelayakan hari-ke-hari per kelompok lokasi (hanya stasiun di aq_daily):")
dd = d.assign(valid=d.pm25_ug_m3.notna())
display(dd.groupby("kelompok_lokasi", observed=True).agg(stasiun=("kode_stasiun", "nunique"), hari=("valid", "size"), persen_hari_sah=("valid", lambda s: round(100 * s.mean(), 1)),
        persen_lewat_bm=("lewat_bm_pm25", lambda s: round(100 * s.dropna().astype(bool).mean(), 1)), persen_lewat_who=("lewat_who_pm25", lambda s: round(100 * s.dropna().astype(bool).mean(), 1))))

Kelulusan aturan 18 jam dan kelayakan hari-ke-hari per kelompok lokasi (hanya stasiun di aq_daily):


,stasiun,hari,persen_hari_sah,persen_lewat_bm,persen_lewat_who
kelompok_lokasi,,,,,
Perkotaan,7,7126,90.9,8.6,100.0
Jalan Raya,2,2036,100.0,17.3,100.0
Industri,4,4072,93.9,36.2,100.0
Gambut,7,7126,95.5,28.0,98.5
Latar Bersih,6,6108,83.4,0.1,19.2


## 8. Penggabungan kesehatan–paparan
`health_aq_daily` = data kesehatan harian (27 RS × 1.018 hari, 2023-12-31 dibuang) + atribut RS/stasiun pasangan + PM2.5 harian stasiun pasangan pada lag 0–7 hari kalender dan rata-rata 3 hari (lag 0–2, harus lengkap).

In [18]:
lag_kol = [f"pm25_lag{L}" for L in cfg.LAG_PM25]
cov = health.groupby(["kode_stasiun", "status_analisis_stasiun"], observed=True).agg(
    hari=("date", "size"), lag0=("pm25_lag0", lambda s: s.notna().sum()), lag0_7_lengkap=("pm25_lag0", "size"), ma3=("pm25_ma3", lambda s: s.notna().sum())).reset_index()
cov["lag0_7_lengkap"] = health.groupby("kode_stasiun")[lag_kol].apply(lambda x: x.notna().all(axis=1).sum()).reindex(cov.kode_stasiun).values
cov["persen_lag0"] = (100 * cov.lag0 / cov.hari).round(1); cov["persen_lag0_7_lengkap"] = (100 * cov.lag0_7_lengkap / cov.hari).round(1)
display(cov.sort_values("persen_lag0_7_lengkap").reset_index(drop=True))
print("Total baris:", rp(len(health)), "| hari dengan paparan lag0:", rp(health.ada_paparan_pm25.sum()), f"({fl(100 * health.ada_paparan_pm25.mean(), 1)}%)",
      "| baris dengan lag 0–7 lengkap:", rp(health[lag_kol].notna().all(axis=1).sum()))
print("\nLag 1–7 pada 7 hari pertama (2024-01-01…) tidak tersedia karena AQMS dimulai 2024-01-01 (disengaja, tidak diimputasi):")
display(health[health.kode_stasiun == "KTG-01"].sort_values("date").head(9)[["date", "pm25_lag0", "pm25_lag1", "pm25_lag2", "pm25_lag7", "pm25_ma3"]].round(1))

,kode_stasiun,status_analisis_stasiun,hari,lag0,lag0_7_lengkap,ma3,persen_lag0,persen_lag0_7_lengkap
0,JMB-02,Dikecualikan,1018,0,0,0,0.0,0.0
1,MER-01,Terbatas,1018,2,0,0,0.2,0.0
2,MDN-01,Terbatas,1018,371,0,62,36.4,0.0
3,PLB-01,Terbatas,1018,696,166,379,68.4,16.3
4,SM2-01,Terbatas,1018,771,646,683,75.7,63.5
5,BGL-01,Layak,1018,1018,1011,1016,100.0,99.3
6,SRG-01,Layak,1018,1018,1011,1016,100.0,99.3
7,PSW-01,Layak,1018,1018,1011,1016,100.0,99.3
8,PLU-01,Layak,1018,1018,1011,1016,100.0,99.3
9,PKU-01,Layak,1018,1018,1011,1016,100.0,99.3


Total baris: 27.486 | hari dengan paparan lag0: 24.236 (88,2%) | baris dengan lag 0–7 lengkap: 23.054

Lag 1–7 pada 7 hari pertama (2024-01-01…) tidak tersedia karena AQMS dimulai 2024-01-01 (disengaja, tidak diimputasi):


,date,pm25_lag0,pm25_lag1,pm25_lag2,pm25_lag7,pm25_ma3
3054,2024-01-01,23.3,NaN,NaN,NaN,NaN
3055,2024-01-02,22.2,23.3,NaN,NaN,NaN
3056,2024-01-03,22.9,22.2,23.3,NaN,22.8
3057,2024-01-04,21.5,22.9,22.2,NaN,22.2
3058,2024-01-05,23.3,21.5,22.9,NaN,22.6
3059,2024-01-06,22.3,23.3,21.5,NaN,22.4
3060,2024-01-07,21.0,22.3,23.3,NaN,22.2
3061,2024-01-08,21.5,21.0,22.3,23.3,21.6
3062,2024-01-09,21.7,21.5,21.0,22.2,21.4


In [19]:
print("Contoh: awal episode kabut asap KTG-01/RS-XXX-04 (lag dihitung per hari kalender di stasiun pasangan):")
x = health[(health.hospital_id == "RS-XXX-04") & (health.date >= "2026-08-30") & (health.date <= "2026-09-06")]
display(x[["date", "pm25_lag0", "pm25_lag1", "pm25_lag7", "pm25_ma3", "kategori_ispu_pm25_hari0", "episode_asap", "resp_admissions_under5", "resp_admissions_over60", "asthma_emergency_cases"]].round(1))
# verifikasi: lag k = PM2.5 stasiun pada tanggal - k
ktg = daily[daily.kode_stasiun == "KTG-01"].set_index("tanggal").pm25_ug_m3
h4 = health[health.hospital_id == "RS-XXX-04"].set_index("date")
for L in (0, 3, 7):
    ref = ktg.shift(L).reindex(h4.index)
    assert ((h4[f"pm25_lag{L}"] == ref) | (h4[f"pm25_lag{L}"].isna() & ref.isna())).all()
print("Lag 0, 3, 7 untuk RS-XXX-04 sama dengan PM2.5 harian KTG-01 yang digeser (lulus)")

Contoh: awal episode kabut asap KTG-01/RS-XXX-04 (lag dihitung per hari kalender di stasiun pasangan):


,date,pm25_lag0,pm25_lag1,pm25_lag7,pm25_ma3,kategori_ispu_pm25_hari0,episode_asap,resp_admissions_under5,resp_admissions_over60,asthma_emergency_cases
4026,2026-08-30,95.5,94.8,94.0,94.5,Tidak Sehat,False,10,14,8
4027,2026-08-31,95.3,95.5,94.8,95.2,Tidak Sehat,False,14,15,7
4028,2026-09-01,420.7,95.3,95.1,203.8,Berbahaya,True,32,27,16
4029,2026-09-02,419.6,420.7,94.0,311.9,Berbahaya,True,35,39,17
4030,2026-09-03,420.2,419.6,95.3,420.2,Berbahaya,True,31,27,22
4031,2026-09-04,419.6,420.2,93.3,419.8,Berbahaya,True,18,23,15
4032,2026-09-05,419.8,419.6,94.8,419.8,Berbahaya,True,20,18,7
4033,2026-09-06,419.0,419.8,95.5,419.4,Berbahaya,True,20,28,14


Lag 0, 3, 7 untuk RS-XXX-04 sama dengan PM2.5 harian KTG-01 yang digeser (lulus)


## 9. Temuan
Sel berikut menyusun temuan **langsung dari tabel di atas** dan menyimpannya ke `reports/findings/03_preprocessing.md`; teks yang sama ditempel sebagai sel markdown terakhir.

In [20]:
def S(metrik, entitas=None, kolom="jumlah"):
    x = ls[ls.metrik == metrik]
    if entitas is not None:
        x = x[x.entitas == entitas]
    return int(x[kolom].sum())
def nst(metrik):
    return int((ls[ls.metrik == metrik].jumlah > 0).sum())
def Q(k, c):
    return qc.set_index("kode_stasiun").loc[k, c]

n_mentah = int(qc.n_baris_mentah.sum())
jam_semua = S("jam_valid_akhir", kolom="sesudah")
n_valid_mentah = int(qc.n_baris_valid_mentah.sum())
ppm_st = sorted(sites[sites.satuan_asli == "ppm"].kode_stasiun)
kecuali = sorted(sites[~sites.dipakai_di_aq_hourly_daily].kode_stasiun)
terbatas = sorted(qc[qc.status_analisis == "Terbatas"].kode_stasiun)
o_ep = d[d.episode_asap]
ep_st = ep.groupby("kode_stasiun", observed=True).agg(n=("id_episode", "nunique"), hari=("hari", "sum"), maks=("pm25_maks_harian", "max"))
bj = ep[ep.kode_stasiun == "BJB-01"]
mer_h = int(qc.set_index("kode_stasiun").loc["MER-01", "n_jam_pm25_valid"])
ketat, longgar = bandingan["aturan ≥18 jam (dipakai)"], bandingan["aturan ≥12 jam (sensitivitas)"]
kemarau_sm2 = 100 * kem_hr.pm25_ug_m3.notna().mean(); kemarau_sm2_l = 100 * kem_hr.pm25_ug_m3_longgar.notna().mean()
berbahaya = int((d.kategori_ispu_pm25 == "Berbahaya").sum())
lulus = int(ulang.lulus.sum())
nm = lambda xs: ", ".join(xs)

tbl_keluaran = pd.DataFrame(info)[["tabel", "baris", "kolom", "grain"]].assign(baris=lambda x: x.baris.map(rp))
tbl_dampak = pd.DataFrame([
    (1, "Teks galat → NaN", S("sel_teks_galat_menjadi_nan"), "sel", nst("sel_teks_galat_menjadi_nan")),
    (2, "ppm → µg/m³", S("sel_dikonversi_ppm_ke_ugm3"), "sel", nst("sel_dikonversi_ppm_ke_ugm3")),
    (3, "Timestamp tak terbaca dibuang", S("baris_timestamp_tak_terbaca"), "baris", nst("baris_timestamp_tak_terbaca")),
    (3, "Timestamp di luar periode dibuang", S("baris_di_luar_periode"), "baris", nst("baris_di_luar_periode")),
    (3, "Dibulatkan ke jam terdekat", S("baris_dibulatkan_ke_jam_terdekat"), "baris", nst("baris_dibulatkan_ke_jam_terdekat")),
    (3, "Bacaan sub-jam → floor jam penuh", S("baris_subjam_dipotong_ke_jam_penuh"), "baris", nst("baris_subjam_dipotong_ke_jam_penuh")),
    (4, "Sentinel → NaN", S("sel_sentinel_menjadi_nan"), "sel", nst("sel_sentinel_menjadi_nan")),
    (4, "Polutan negatif → NaN", S("sel_negatif_menjadi_nan"), "sel", nst("sel_negatif_menjadi_nan")),
    (4, "Cuaca di luar batas fisik → NaN", S("sel_di_luar_batas_fisik_cuaca_menjadi_nan"), "sel", nst("sel_di_luar_batas_fisik_cuaca_menjadi_nan")),
    (4, "Baris VALID ber-PM ekstrem (>500) DIPERTAHANKAN", S("baris_valid_pm_ekstrem_dipertahankan"), "baris", nst("baris_valid_pm_ekstrem_dipertahankan")),
    (5, "Baris non-VALID dibuang", S("jumlah_baris_non_valid"), "baris", nst("jumlah_baris_non_valid")),
    (5, "Baris digabung (rata-rata per jam)", S("jumlah_baris_per_jam"), "baris", nst("jumlah_baris_per_jam")),
    (6, "Hari ditolak (< 18 jam valid)", S("hari_pm25_ditolak_kurang_dari_18_jam"), "hari", nst("hari_pm25_ditolak_kurang_dari_18_jam")),
    (7, "Episode kabut asap terdeteksi", S("episode_kabut_asap"), "episode", nst("episode_kabut_asap")),
], columns=["langkah", "aturan", "jumlah", "satuan", "stasiun_terdampak"]).assign(jumlah=lambda x: x.jumlah.map(rp))

tbl_stasiun = qc[qc.kode_stasiun.isin(["JMB-02", "MER-01", "MDN-01", "PLB-01", "SM2-01", "BJB-01"])][
    ["kode_stasiun", "status_sensor", "persen_jam_pm25_valid", "persen_hari_pm25_valid", "persen_hari_pm25_valid_kemarau", "status_analisis"]].round(1)
tbl_ppm = vsat.groupby("kode_stasiun").agg(rasio_min=("rasio", "min"), rasio_maks=("rasio", "max"), rasio_pm25_vs_baseline=("rasio_vs_baseline", "max"),
                                            tanpa_konversi_maks=("rasio_tanpa_konversi", "max")).round(3).reset_index()

temuan = f"""# 03 — Preprocessing: Temuan
> Dihasilkan otomatis oleh `notebooks/03_preprocessing.ipynb` dari keluaran `etl/run_preprocessing.py`. Semua angka berasal dari `data/processed/*.parquet` dan `reports/preprocessing_log.csv`.
> Rincian setiap langkah (per stasiun) ada di `reports/preprocessing_log.csv`. Data mentah di `data/raw/` tidak diubah.

## 1. Ringkasan keluaran
{md_t(tbl_keluaran)}

Alur baris: **{rp(n_mentah)}** baris mentah (27 stasiun) → **{rp(n_valid_mentah)}** baris `VALID` → **{rp(jam_semua)}** jam valid (27 stasiun) → **{rp(len(hourly))}** jam pada `aq_hourly` ({len(sites) - len(kecuali)} stasiun; {nm(kecuali)} dikecualikan) → **{rp(len(daily))}** hari-stasiun pada `aq_daily` (grid kalender lengkap, {rp(daily.pm25_ug_m3.notna().sum())} hari dengan PM2.5 sah).

## 2. Dampak tiap langkah (seluruh stasiun)
{md_t(tbl_dampak)}

## 3. Perlakuan untuk stasiun bermasalah
{md_t(tbl_stasiun, 1)}

**JMB-02 (Muaro Jambi): dikecualikan** dari `aq_hourly`, `aq_daily`, dan paparan kesehatan. Hanya {int(prof02.loc["JMB-02", "n_valid"])} dari {rp(prof02.loc["JMB-02", "n_baris"])} baris `VALID` ({fl(prof02.loc["JMB-02", "persen_valid"], 2)}%), tidak ada baris VALID setelah {tj.index.max()}, {fl(prof02.loc["JMB-02", "n_baris_mustahil"] / prof02.loc["JMB-02", "n_baris"] * 100, 1)}% baris berisi nilai mustahil, dan {int((hj >= 18).sum())} hari yang memenuhi ≥18 jam VALID. Sensor berstatus Defective. Tidak diimputasi dan tidak ada aturan yang dilonggarkan. Stasiun tetap ada di `sites_clean` dan `qc_summary` sebagai bukti kebutuhan perbaikan jaringan (Tujuan 5); RS-XXX-27 tetap ada di `health_aq_daily` dengan paparan kosong.

**MER-01 (Merauke): Terbatas.** {rp(mer_h)} jam valid ({fl(Q("MER-01", "persen_jam_pm25_valid"), 1)}%) tersebar acak dalam {rp(blok)} blok pendek; hanya {int(ambang.loc[0, "hari_memenuhi"])} hari yang memenuhi ≥18 jam ({int(ambang.loc[1, "hari_memenuhi"])} hari bila ambang 12 jam). Nilai VALID-nya wajar dibanding stasiun latar (median PM2.5 {fl(hourly[hourly.kode_stasiun == "MER-01"].pm25_ug_m3.median(), 1)} µg/m³), sehingga **data per jam dipertahankan** untuk profil diurnal/klimatologi dan bukti kualitas jaringan, tetapi **tidak dipakai untuk model harian/efek jeda kesehatan**. Aturan 75% tidak dilonggarkan karena rata-rata dari ±6 jam tidak mewakili satu hari. Jam valid tersebar merata antar-bulan ({jb.min()}–{jb.max()} jam/bulan), jadi tidak ada bukti bias musiman, namun puncak asap tidak mungkin tertangkap.

**Stasiun Terbatas lainnya** ({nm([k for k in terbatas if k != "MER-01"])}) tetap berada di tabel dengan peringatan; gunakan `status_analisis` untuk menyaringnya pada model.
- **MDN-01**: {fl(Q("MDN-01", "persen_jam_pm25_valid"), 1)}% jam valid ({rp(cfg.EXPECTED_HOURS - Q("MDN-01", "n_baris_mentah"))} jam tidak tercatat di sumber; gap terpanjang {int(Q("MDN-01", "gap_terpanjang_jam"))} jam setelah pembersihan) dan hanya {fl(Q("MDN-01", "persen_hari_pm25_valid"), 1)}% hari sah.
- **PLB-01**: cakupan jam {fl(Q("PLB-01", "persen_jam_pm25_valid"), 1)}% tetapi hari sah hanya {fl(Q("PLB-01", "persen_hari_pm25_valid"), 1)}% (jam `MISSING` tersebar); logger 15 menit pada Sep–Okt dirata-rata per jam ({rp(S("baris_subjam_dipotong_ke_jam_penuh", "PLB-01"))} baris), bukan dihapus sebagai duplikat.
- **SM2-01**: hari sah {fl(Q("SM2-01", "persen_hari_pm25_valid"), 1)}% secara umum, tetapi hanya **{fl(kemarau_sm2, 0)}%** pada Jul–Okt (vs {fl(kemarau_sm2_l, 0)}% dengan ambang 12 jam), karena nilai saturasi 999 (`SUSPECT`) dibuang. Deteksi episode: {int(ketat["episode"])} episode / {int(ketat["hari_episode"])} hari (ambang 18 jam) vs {int(longgar["episode"])} episode / {int(longgar["hari_episode"])} hari (ambang 12 jam). Puncak asap SM2-01 hampir pasti **terpangkas (bias ke bawah)**.
- **BJB-01**: 100% VALID dan lolos validasi satuan, tetapi pola musimnya menyimpang dari stasiun gambut lain (lihat notebook 02) dan menghasilkan **satu episode {int(bj.hari.iloc[0]) if len(bj) else 0} hari** ({bj.awal.iloc[0]:%Y-%m-%d}…{bj.akhir.iloc[0]:%Y-%m-%d}) yang tidak lazim; dibiarkan apa adanya dan ditandai untuk EDA.

## 4. Validasi
- **{lulus}/{len(ulang)}** pemeriksaan lulus (dijalankan di pipeline dan diulang di notebook): tidak ada duplikat (stasiun, jam), (stasiun, hari), maupun (RS, hari); semua polutan ≥ 0; semua kolom polutan `*_ug_m3` tanpa `*_ppm`; cuaca dalam batas fisik; rata-rata harian hanya dari ≥18 jam; puncak PM > 500 pada baris VALID dipertahankan ({rp(S("baris_valid_pm_ekstrem_dipertahankan"))} baris; PM2.5 KTG-01 maks {fl(hourly[hourly.kode_stasiun == "KTG-01"].pm25_ug_m3.max(), 1)} µg/m³ masih ada).
- Rekonsiliasi baris per stasiun cocok untuk seluruh {len(rek)} stasiun: mentah = tak terbaca + luar periode + non-VALID + VALID; VALID − digabung = jam akhir.
- **Konversi ppm** ({nm(ppm_st)}): median PM2.5 setelah ×1000 sebanding dengan `baseline_pm25` (rasio {fl(vsat.rasio_vs_baseline.min(), 2)}–{fl(vsat.rasio_vs_baseline.max(), 2)}); seluruh polutan sebanding stasiun sejenis (rasio {fl(vsat.rasio.min(), 2)}–{fl(vsat.rasio.max(), 2)}), sedangkan tanpa konversi rasionya hanya {fl(vsat.rasio_tanpa_konversi.min(), 4)}–{fl(vsat.rasio_tanpa_konversi.max(), 4)}.

{md_t(tbl_ppm, 3)}

## 5. Dampak pada data analisis
- **Sebelum vs sesudah.** PM2.5 mentah memuat nilai hingga {fl(sb["maks"].max(), 0)} dan {rp(sb["n_negatif"].sum())} nilai negatif (termasuk sentinel -999/-9999); sesudah pembersihan rentang antar-stasiun {fl(sa["median"].min(), 1)}–{fl(sa["median"].max(), 1)} µg/m³ (median) dengan maksimum {fl(sa["maks"].max(), 1)} µg/m³. Median stasiun ppm naik dari {fl(sb.loc[ppm_st, "median"].min(), 3)}–{fl(sb.loc[ppm_st, "median"].max(), 3)} (label ppm) menjadi {fl(sa.loc[ppm_st, "median"].min(), 1)}–{fl(sa.loc[ppm_st, "median"].max(), 1)} µg/m³.
- **Kategori ISPU PM2.5** ({rp(kat.sum())} hari sah): """ + ", ".join(f"{k} {fl(100 * v / kat.sum(), 1)}%" for k, v in kat.items()) + f""". {berbahaya} hari berkategori Berbahaya.
- **Episode kabut asap** (PM2.5 harian > 150 µg/m³, ≥ 3 hari berturut-turut): {int(ep_st.n.sum())} episode, {int(ep_st.hari.sum())} hari pada {len(ep_st)} stasiun ({nm(sorted(ep_st.index.astype(str)))}); episode dengan PM2.5 harian tertinggi: """ + "; ".join(f"{k} {int(r.hari)} hari, maks harian {fl(r.maks, 0)} µg/m³" for k, r in ep_st.sort_values("maks", ascending=False).head(3).iterrows()) + f""". Episode hanya dibangun dari hari dengan rata-rata sah, sehingga stasiun dengan cakupan rendah (SM2-01, PLB-01, MDN-01) cenderung **kurang mendeteksi** episode.
- **Paparan kesehatan.** {rp(len(health))} baris RS-hari; PM2.5 lag 0 tersedia pada {fl(100 * health.ada_paparan_pm25.mean(), 1)}% baris dan lag 0–7 lengkap pada {fl(100 * health[lag_kol].notna().all(axis=1).mean(), 1)}%. Baris tanpa paparan dibiarkan `NaN` (tidak diimputasi) dan dapat disaring dengan `ada_paparan_pm25` / `status_analisis_stasiun`.

## 6. Keputusan, asumsi, dan perbedaan dengan dokumen sebelumnya
1. **Musim kemarau = Mei–Oktober** (ketentuan proyek), lebih lebar daripada Jun–Okt pada `01_business_understanding.md` (asumsi A5). Seluruh fitur memakai definisi Mei–Okt.
2. **Kelompok lokasi** memakai nama "Jalan Raya" (sebelumnya "Tepi Jalan") agar konsisten dengan ketentuan proyek; `src/config.py` dan notebook 02 diperbarui.
3. **Tanggal harian memakai WIB** sesuai pencatatan sumber; `jam_lokal` (WITA/WIT) disediakan hanya untuk membaca pola harian. Tanggal kasus kesehatan diasumsikan sejajar dengan tanggal WIB (selisih ≤ 2 jam untuk stasiun WITA/WIT).
4. **ISPU hanya untuk PM2.5** (breakpoint dari CLAUDE.md); parameter lain belum dihitung karena breakpoint belum diverifikasi terhadap naskah Permen LHK 14/2020.
5. **Rata-rata 3 hari** (`pm25_ma3`) mensyaratkan 3 hari lengkap (tanpa `min_periods` longgar); lag dihitung per hari **kalender** pada grid lengkap sehingga hari tanpa data menjadi `NaN`, bukan menggeser baris.
6. **Status analisis** (Layak ≥ 75% pada jam, hari, dan hari kemarau; Terbatas; Dikecualikan < 10% jam) adalah ambang praktis proyek, bukan standar.
7. **Arah angin** dirata-rata secara melingkar; rata-rata harian arah angin juga melingkar.
8. Baris kesehatan 2023-12-31 ({int(log[log.metrik == "baris_sebelum_periode_aqms_dibuang"].jumlah.sum())} baris) dibuang karena tidak punya pasangan paparan.

## 7. Keterbatasan
- Hanya baris `VALID` yang dipakai; kebijakan ini memangkas puncak asap pada SM2-01 (dan mungkin sensor lain) sehingga estimasi dampak kabut asap bias ke bawah. Kolom `pm25_ug_m3_longgar` hanya untuk sensitivitas.
- Konversi PM "ppm" → µg/m³ (×1000) adalah hipotesis yang tervalidasi lewat orde besaran, bukan lewat pengukuran referensi.
- `pm_ekstrem` (PM > 500 µg/m³) hanya penanda untuk ditelaah; nilai ekstrem asli tidak dibuang, tetapi tidak dapat dibedakan secara pasti dari galat alat tanpa data pendukung (CO, hotspot).
- Stasiun BJB-01 dan SM2-01 memiliki PM2.5 tinggi namun kasus RS rendah; pembersihan tidak mengubah hal ini (lihat notebook 02), sehingga hasil hubungan PM2.5–kesehatan harus dilaporkan dengan sensitivitas tanpa stasiun tersebut.
- Tidak ada inferensi kausal pada tahap ini.

## 8. Cara menjalankan ulang
`python etl/run_preprocessing.py` membangun ulang seluruh keluaran (± 30 detik, deterministik). Skrip berhenti tanpa menulis apa pun bila satu validasi gagal.
"""
(cfg.FINDINGS_DIR / "03_preprocessing.md").write_text(temuan, encoding="utf-8")
print(f"Tersimpan: reports/findings/03_preprocessing.md ({len(temuan.splitlines())} baris)")
display(Markdown(temuan))

Tersimpan: reports/findings/03_preprocessing.md (89 baris)


# 03 — Preprocessing: Temuan
> Dihasilkan otomatis oleh `notebooks/03_preprocessing.ipynb` dari keluaran `etl/run_preprocessing.py`. Semua angka berasal dari `data/processed/*.parquet` dan `reports/preprocessing_log.csv`.
> Rincian setiap langkah (per stasiun) ada di `reports/preprocessing_log.csv`. Data mentah di `data/raw/` tidak diubah.

## 1. Ringkasan keluaran
| tabel           | baris   | kolom   | grain                                                     |
|:----------------|:--------|:--------|:----------------------------------------------------------|
| aq_hourly       | 591.047 | 24      | 1 baris = stasiun × jam (hanya jam dengan ≥1 baris VALID) |
| aq_daily        | 26.468  | 34      | 1 baris = stasiun × hari (grid kalender lengkap)          |
| health_aq_daily | 27.486  | 47      | 1 baris = RS × hari + paparan PM2.5 stasiun pasangan      |
| sites_clean     | 27      | 34      | 1 baris = stasiun (sites + sensors + hospitals)           |
| qc_summary      | 27      | 35      | 1 baris = stasiun (kualitas & status analisis)            |

Alur baris: **667.981** baris mentah (27 stasiun) → **599.185** baris `VALID` → **591.168** jam valid (27 stasiun) → **591.047** jam pada `aq_hourly` (26 stasiun; JMB-02 dikecualikan) → **26.468** hari-stasiun pada `aq_daily` (grid kalender lengkap, 24.236 hari dengan PM2.5 sah).

## 2. Dampak tiap langkah (seluruh stasiun)
| langkah   | aturan                                          | jumlah   | satuan   | stasiun_terdampak   |
|:----------|:------------------------------------------------|:---------|:---------|:--------------------|
| 1         | Teks galat → NaN                                | 43.171   | sel      | 1                   |
| 2         | ppm → µg/m³                                     | 414.240  | sel      | 3                   |
| 3         | Timestamp tak terbaca dibuang                   | 0        | baris    | 0                   |
| 3         | Timestamp di luar periode dibuang               | 132      | baris    | 1                   |
| 3         | Dibulatkan ke jam terdekat                      | 411.488  | baris    | 17                  |
| 3         | Bacaan sub-jam → floor jam penuh                | 16.025   | baris    | 2                   |
| 4         | Sentinel → NaN                                  | 43.185   | sel      | 2                   |
| 4         | Polutan negatif → NaN                           | 36.830   | sel      | 2                   |
| 4         | Cuaca di luar batas fisik → NaN                 | 59.659   | sel      | 2                   |
| 4         | Baris VALID ber-PM ekstrem (>500) DIPERTAHANKAN | 2.549    | baris    | 4                   |
| 5         | Baris non-VALID dibuang                         | 68.665   | baris    | 26                  |
| 5         | Baris digabung (rata-rata per jam)              | 8.016    | baris    | 1                   |
| 6         | Hari ditolak (< 18 jam valid)                   | 1.996    | hari     | 4                   |
| 7         | Episode kabut asap terdeteksi                   | 21       | episode  | 7                   |

## 3. Perlakuan untuk stasiun bermasalah
| kode_stasiun   | status_sensor   | persen_jam_pm25_valid   | persen_hari_pm25_valid   | persen_hari_pm25_valid_kemarau   | status_analisis   |
|:---------------|:----------------|:------------------------|:-------------------------|:---------------------------------|:------------------|
| BJB-01         | Active          | 100,0                   | 100,0                    | 100,0                            | Layak             |
| JMB-02         | Defective       | 0,5                     | –                        | –                                | Dikecualikan      |
| MDN-01         | Intermittent    | 68,1                    | 36,4                     | 37,2                             | Terbatas          |
| MER-01         | Intermittent    | 20,0                    | 0,2                      | 0,4                              | Terbatas          |
| PLB-01         | Intermittent    | 79,5                    | 68,4                     | 72,3                             | Terbatas          |
| SM2-01         | Active          | 85,0                    | 75,7                     | 53,8                             | Terbatas          |

**JMB-02 (Muaro Jambi): dikecualikan** dari `aq_hourly`, `aq_daily`, dan paparan kesehatan. Hanya 122 dari 24.432 baris `VALID` (0,50%), tidak ada baris VALID setelah 2024-08, 99,5% baris berisi nilai mustahil, dan 0 hari yang memenuhi ≥18 jam VALID. Sensor berstatus Defective. Tidak diimputasi dan tidak ada aturan yang dilonggarkan. Stasiun tetap ada di `sites_clean` dan `qc_summary` sebagai bukti kebutuhan perbaikan jaringan (Tujuan 5); RS-XXX-27 tetap ada di `health_aq_daily` dengan paparan kosong.

**MER-01 (Merauke): Terbatas.** 4.891 jam valid (20,0%) tersebar acak dalam 1.837 blok pendek; hanya 2 hari yang memenuhi ≥18 jam (54 hari bila ambang 12 jam). Nilai VALID-nya wajar dibanding stasiun latar (median PM2.5 16,1 µg/m³), sehingga **data per jam dipertahankan** untuk profil diurnal/klimatologi dan bukti kualitas jaringan, tetapi **tidak dipakai untuk model harian/efek jeda kesehatan**. Aturan 75% tidak dilonggarkan karena rata-rata dari ±6 jam tidak mewakili satu hari. Jam valid tersebar merata antar-bulan (66–213 jam/bulan), jadi tidak ada bukti bias musiman, namun puncak asap tidak mungkin tertangkap.

**Stasiun Terbatas lainnya** (MDN-01, PLB-01, SM2-01) tetap berada di tabel dengan peringatan; gunakan `status_analisis` untuk menyaringnya pada model.
- **MDN-01**: 68,1% jam valid (3.632 jam tidak tercatat di sumber; gap terpanjang 122 jam setelah pembersihan) dan hanya 36,4% hari sah.
- **PLB-01**: cakupan jam 79,5% tetapi hari sah hanya 68,4% (jam `MISSING` tersebar); logger 15 menit pada Sep–Okt dirata-rata per jam (15.933 baris), bukan dihapus sebagai duplikat.
- **SM2-01**: hari sah 75,7% secara umum, tetapi hanya **30%** pada Jul–Okt (vs 97% dengan ambang 12 jam), karena nilai saturasi 999 (`SUSPECT`) dibuang. Deteksi episode: 6 episode / 20 hari (ambang 18 jam) vs 28 episode / 223 hari (ambang 12 jam). Puncak asap SM2-01 hampir pasti **terpangkas (bias ke bawah)**.
- **BJB-01**: 100% VALID dan lolos validasi satuan, tetapi pola musimnya menyimpang dari stasiun gambut lain (lihat notebook 02) dan menghasilkan **satu episode 204 hari** (2024-09-07…2025-03-29) yang tidak lazim; dibiarkan apa adanya dan ditandai untuk EDA.

## 4. Validasi
- **21/21** pemeriksaan lulus (dijalankan di pipeline dan diulang di notebook): tidak ada duplikat (stasiun, jam), (stasiun, hari), maupun (RS, hari); semua polutan ≥ 0; semua kolom polutan `*_ug_m3` tanpa `*_ppm`; cuaca dalam batas fisik; rata-rata harian hanya dari ≥18 jam; puncak PM > 500 pada baris VALID dipertahankan (2.549 baris; PM2.5 KTG-01 maks 1.620,5 µg/m³ masih ada).
- Rekonsiliasi baris per stasiun cocok untuk seluruh 27 stasiun: mentah = tak terbaca + luar periode + non-VALID + VALID; VALID − digabung = jam akhir.
- **Konversi ppm** (BJB-01, MDN-01, PSW-01): median PM2.5 setelah ×1000 sebanding dengan `baseline_pm25` (rasio 0,80–1,29); seluruh polutan sebanding stasiun sejenis (rasio 0,65–2,04), sedangkan tanpa konversi rasionya hanya 0,0003–0,0016.

| kode_stasiun   | rasio_min   | rasio_maks   | rasio_pm25_vs_baseline   | tanpa_konversi_maks   |
|:---------------|:------------|:-------------|:-------------------------|:----------------------|
| BJB-01         | 1,165       | 1,705        | 1,289                    | 0,002                 |
| MDN-01         | 0,899       | 2,039        | 1,086                    | 0,002                 |
| PSW-01         | 0,654       | 1,445        | 0,801                    | 0,001                 |

## 5. Dampak pada data analisis
- **Sebelum vs sesudah.** PM2.5 mentah memuat nilai hingga 99.954 dan 9.369 nilai negatif (termasuk sentinel -999/-9999); sesudah pembersihan rentang antar-stasiun 8,9–58,9 µg/m³ (median) dengan maksimum 1.620,5 µg/m³. Median stasiun ppm naik dari 0,043–0,047 (label ppm) menjadi 43,5–46,5 µg/m³.
- **Kategori ISPU PM2.5** (24.236 hari sah): Baik 17,7%, Sedang 65,1%, Tidak Sehat 15,2%, Sangat Tidak Sehat 0,7%, Berbahaya 1,2%. 301 hari berkategori Berbahaya.
- **Episode kabut asap** (PM2.5 harian > 150 µg/m³, ≥ 3 hari berturut-turut): 21 episode, 396 hari pada 7 stasiun (BJB-01, KTG-01, KTG-02, PKN-01, PLB-01, SM2-01, STG-01); episode dengan PM2.5 harian tertinggi: KTG-01 44 hari, maks harian 1.128 µg/m³; KTG-02 44 hari, maks harian 1.120 µg/m³; PKN-01 6 hari, maks harian 628 µg/m³. Episode hanya dibangun dari hari dengan rata-rata sah, sehingga stasiun dengan cakupan rendah (SM2-01, PLB-01, MDN-01) cenderung **kurang mendeteksi** episode.
- **Paparan kesehatan.** 27.486 baris RS-hari; PM2.5 lag 0 tersedia pada 88,2% baris dan lag 0–7 lengkap pada 83,9%. Baris tanpa paparan dibiarkan `NaN` (tidak diimputasi) dan dapat disaring dengan `ada_paparan_pm25` / `status_analisis_stasiun`.

## 6. Keputusan, asumsi, dan perbedaan dengan dokumen sebelumnya
1. **Musim kemarau = Mei–Oktober** (ketentuan proyek), lebih lebar daripada Jun–Okt pada `01_business_understanding.md` (asumsi A5). Seluruh fitur memakai definisi Mei–Okt.
2. **Kelompok lokasi** memakai nama "Jalan Raya" (sebelumnya "Tepi Jalan") agar konsisten dengan ketentuan proyek; `src/config.py` dan notebook 02 diperbarui.
3. **Tanggal harian memakai WIB** sesuai pencatatan sumber; `jam_lokal` (WITA/WIT) disediakan hanya untuk membaca pola harian. Tanggal kasus kesehatan diasumsikan sejajar dengan tanggal WIB (selisih ≤ 2 jam untuk stasiun WITA/WIT).
4. **ISPU hanya untuk PM2.5** (breakpoint dari CLAUDE.md); parameter lain belum dihitung karena breakpoint belum diverifikasi terhadap naskah Permen LHK 14/2020.
5. **Rata-rata 3 hari** (`pm25_ma3`) mensyaratkan 3 hari lengkap (tanpa `min_periods` longgar); lag dihitung per hari **kalender** pada grid lengkap sehingga hari tanpa data menjadi `NaN`, bukan menggeser baris.
6. **Status analisis** (Layak ≥ 75% pada jam, hari, dan hari kemarau; Terbatas; Dikecualikan < 10% jam) adalah ambang praktis proyek, bukan standar.
7. **Arah angin** dirata-rata secara melingkar; rata-rata harian arah angin juga melingkar.
8. Baris kesehatan 2023-12-31 (5 baris) dibuang karena tidak punya pasangan paparan.

## 7. Keterbatasan
- Hanya baris `VALID` yang dipakai; kebijakan ini memangkas puncak asap pada SM2-01 (dan mungkin sensor lain) sehingga estimasi dampak kabut asap bias ke bawah. Kolom `pm25_ug_m3_longgar` hanya untuk sensitivitas.
- Konversi PM "ppm" → µg/m³ (×1000) adalah hipotesis yang tervalidasi lewat orde besaran, bukan lewat pengukuran referensi.
- `pm_ekstrem` (PM > 500 µg/m³) hanya penanda untuk ditelaah; nilai ekstrem asli tidak dibuang, tetapi tidak dapat dibedakan secara pasti dari galat alat tanpa data pendukung (CO, hotspot).
- Stasiun BJB-01 dan SM2-01 memiliki PM2.5 tinggi namun kasus RS rendah; pembersihan tidak mengubah hal ini (lihat notebook 02), sehingga hasil hubungan PM2.5–kesehatan harus dilaporkan dengan sensitivitas tanpa stasiun tersebut.
- Tidak ada inferensi kausal pada tahap ini.

## 8. Cara menjalankan ulang
`python etl/run_preprocessing.py` membangun ulang seluruh keluaran (± 30 detik, deterministik). Skrip berhenti tanpa menulis apa pun bila satu validasi gagal.


# 03 — Preprocessing: Temuan
> Dihasilkan otomatis oleh `notebooks/03_preprocessing.ipynb` dari keluaran `etl/run_preprocessing.py`. Semua angka berasal dari `data/processed/*.parquet` dan `reports/preprocessing_log.csv`.
> Rincian setiap langkah (per stasiun) ada di `reports/preprocessing_log.csv`. Data mentah di `data/raw/` tidak diubah.

## 1. Ringkasan keluaran
| tabel           | baris   | kolom   | grain                                                     |
|:----------------|:--------|:--------|:----------------------------------------------------------|
| aq_hourly       | 591.047 | 24      | 1 baris = stasiun × jam (hanya jam dengan ≥1 baris VALID) |
| aq_daily        | 26.468  | 34      | 1 baris = stasiun × hari (grid kalender lengkap)          |
| health_aq_daily | 27.486  | 47      | 1 baris = RS × hari + paparan PM2.5 stasiun pasangan      |
| sites_clean     | 27      | 34      | 1 baris = stasiun (sites + sensors + hospitals)           |
| qc_summary      | 27      | 35      | 1 baris = stasiun (kualitas & status analisis)            |

Alur baris: **667.981** baris mentah (27 stasiun) → **599.185** baris `VALID` → **591.168** jam valid (27 stasiun) → **591.047** jam pada `aq_hourly` (26 stasiun; JMB-02 dikecualikan) → **26.468** hari-stasiun pada `aq_daily` (grid kalender lengkap, 24.236 hari dengan PM2.5 sah).

## 2. Dampak tiap langkah (seluruh stasiun)
| langkah   | aturan                                          | jumlah   | satuan   | stasiun_terdampak   |
|:----------|:------------------------------------------------|:---------|:---------|:--------------------|
| 1         | Teks galat → NaN                                | 43.171   | sel      | 1                   |
| 2         | ppm → µg/m³                                     | 414.240  | sel      | 3                   |
| 3         | Timestamp tak terbaca dibuang                   | 0        | baris    | 0                   |
| 3         | Timestamp di luar periode dibuang               | 132      | baris    | 1                   |
| 3         | Dibulatkan ke jam terdekat                      | 411.488  | baris    | 17                  |
| 3         | Bacaan sub-jam → floor jam penuh                | 16.025   | baris    | 2                   |
| 4         | Sentinel → NaN                                  | 43.185   | sel      | 2                   |
| 4         | Polutan negatif → NaN                           | 36.830   | sel      | 2                   |
| 4         | Cuaca di luar batas fisik → NaN                 | 59.659   | sel      | 2                   |
| 4         | Baris VALID ber-PM ekstrem (>500) DIPERTAHANKAN | 2.549    | baris    | 4                   |
| 5         | Baris non-VALID dibuang                         | 68.665   | baris    | 26                  |
| 5         | Baris digabung (rata-rata per jam)              | 8.016    | baris    | 1                   |
| 6         | Hari ditolak (< 18 jam valid)                   | 1.996    | hari     | 4                   |
| 7         | Episode kabut asap terdeteksi                   | 21       | episode  | 7                   |

## 3. Perlakuan untuk stasiun bermasalah
| kode_stasiun   | status_sensor   | persen_jam_pm25_valid   | persen_hari_pm25_valid   | persen_hari_pm25_valid_kemarau   | status_analisis   |
|:---------------|:----------------|:------------------------|:-------------------------|:---------------------------------|:------------------|
| BJB-01         | Active          | 100,0                   | 100,0                    | 100,0                            | Layak             |
| JMB-02         | Defective       | 0,5                     | –                        | –                                | Dikecualikan      |
| MDN-01         | Intermittent    | 68,1                    | 36,4                     | 37,2                             | Terbatas          |
| MER-01         | Intermittent    | 20,0                    | 0,2                      | 0,4                              | Terbatas          |
| PLB-01         | Intermittent    | 79,5                    | 68,4                     | 72,3                             | Terbatas          |
| SM2-01         | Active          | 85,0                    | 75,7                     | 53,8                             | Terbatas          |

**JMB-02 (Muaro Jambi): dikecualikan** dari `aq_hourly`, `aq_daily`, dan paparan kesehatan. Hanya 122 dari 24.432 baris `VALID` (0,50%), tidak ada baris VALID setelah 2024-08, 99,5% baris berisi nilai mustahil, dan 0 hari yang memenuhi ≥18 jam VALID. Sensor berstatus Defective. Tidak diimputasi dan tidak ada aturan yang dilonggarkan. Stasiun tetap ada di `sites_clean` dan `qc_summary` sebagai bukti kebutuhan perbaikan jaringan (Tujuan 5); RS-XXX-27 tetap ada di `health_aq_daily` dengan paparan kosong.

**MER-01 (Merauke): Terbatas.** 4.891 jam valid (20,0%) tersebar acak dalam 1.837 blok pendek; hanya 2 hari yang memenuhi ≥18 jam (54 hari bila ambang 12 jam). Nilai VALID-nya wajar dibanding stasiun latar (median PM2.5 16,1 µg/m³), sehingga **data per jam dipertahankan** untuk profil diurnal/klimatologi dan bukti kualitas jaringan, tetapi **tidak dipakai untuk model harian/efek jeda kesehatan**. Aturan 75% tidak dilonggarkan karena rata-rata dari ±6 jam tidak mewakili satu hari. Jam valid tersebar merata antar-bulan (66–213 jam/bulan), jadi tidak ada bukti bias musiman, namun puncak asap tidak mungkin tertangkap.

**Stasiun Terbatas lainnya** (MDN-01, PLB-01, SM2-01) tetap berada di tabel dengan peringatan; gunakan `status_analisis` untuk menyaringnya pada model.
- **MDN-01**: 68,1% jam valid (3.632 jam tidak tercatat di sumber; gap terpanjang 122 jam setelah pembersihan) dan hanya 36,4% hari sah.
- **PLB-01**: cakupan jam 79,5% tetapi hari sah hanya 68,4% (jam `MISSING` tersebar); logger 15 menit pada Sep–Okt dirata-rata per jam (15.933 baris), bukan dihapus sebagai duplikat.
- **SM2-01**: hari sah 75,7% secara umum, tetapi hanya **30%** pada Jul–Okt (vs 97% dengan ambang 12 jam), karena nilai saturasi 999 (`SUSPECT`) dibuang. Deteksi episode: 6 episode / 20 hari (ambang 18 jam) vs 28 episode / 223 hari (ambang 12 jam). Puncak asap SM2-01 hampir pasti **terpangkas (bias ke bawah)**.
- **BJB-01**: 100% VALID dan lolos validasi satuan, tetapi pola musimnya menyimpang dari stasiun gambut lain (lihat notebook 02) dan menghasilkan **satu episode 204 hari** (2024-09-07…2025-03-29) yang tidak lazim; dibiarkan apa adanya dan ditandai untuk EDA.

## 4. Validasi
- **21/21** pemeriksaan lulus (dijalankan di pipeline dan diulang di notebook): tidak ada duplikat (stasiun, jam), (stasiun, hari), maupun (RS, hari); semua polutan ≥ 0; semua kolom polutan `*_ug_m3` tanpa `*_ppm`; cuaca dalam batas fisik; rata-rata harian hanya dari ≥18 jam; puncak PM > 500 pada baris VALID dipertahankan (2.549 baris; PM2.5 KTG-01 maks 1.620,5 µg/m³ masih ada).
- Rekonsiliasi baris per stasiun cocok untuk seluruh 27 stasiun: mentah = tak terbaca + luar periode + non-VALID + VALID; VALID − digabung = jam akhir.
- **Konversi ppm** (BJB-01, MDN-01, PSW-01): median PM2.5 setelah ×1000 sebanding dengan `baseline_pm25` (rasio 0,80–1,29); seluruh polutan sebanding stasiun sejenis (rasio 0,65–2,04), sedangkan tanpa konversi rasionya hanya 0,0003–0,0016.

| kode_stasiun   | rasio_min   | rasio_maks   | rasio_pm25_vs_baseline   | tanpa_konversi_maks   |
|:---------------|:------------|:-------------|:-------------------------|:----------------------|
| BJB-01         | 1,165       | 1,705        | 1,289                    | 0,002                 |
| MDN-01         | 0,899       | 2,039        | 1,086                    | 0,002                 |
| PSW-01         | 0,654       | 1,445        | 0,801                    | 0,001                 |

## 5. Dampak pada data analisis
- **Sebelum vs sesudah.** PM2.5 mentah memuat nilai hingga 99.954 dan 9.369 nilai negatif (termasuk sentinel -999/-9999); sesudah pembersihan rentang antar-stasiun 8,9–58,9 µg/m³ (median) dengan maksimum 1.620,5 µg/m³. Median stasiun ppm naik dari 0,043–0,047 (label ppm) menjadi 43,5–46,5 µg/m³.
- **Kategori ISPU PM2.5** (24.236 hari sah): Baik 17,7%, Sedang 65,1%, Tidak Sehat 15,2%, Sangat Tidak Sehat 0,7%, Berbahaya 1,2%. 301 hari berkategori Berbahaya.
- **Episode kabut asap** (PM2.5 harian > 150 µg/m³, ≥ 3 hari berturut-turut): 21 episode, 396 hari pada 7 stasiun (BJB-01, KTG-01, KTG-02, PKN-01, PLB-01, SM2-01, STG-01); episode dengan PM2.5 harian tertinggi: KTG-01 44 hari, maks harian 1.128 µg/m³; KTG-02 44 hari, maks harian 1.120 µg/m³; PKN-01 6 hari, maks harian 628 µg/m³. Episode hanya dibangun dari hari dengan rata-rata sah, sehingga stasiun dengan cakupan rendah (SM2-01, PLB-01, MDN-01) cenderung **kurang mendeteksi** episode.
- **Paparan kesehatan.** 27.486 baris RS-hari; PM2.5 lag 0 tersedia pada 88,2% baris dan lag 0–7 lengkap pada 83,9%. Baris tanpa paparan dibiarkan `NaN` (tidak diimputasi) dan dapat disaring dengan `ada_paparan_pm25` / `status_analisis_stasiun`.

## 6. Keputusan, asumsi, dan perbedaan dengan dokumen sebelumnya
1. **Musim kemarau = Mei–Oktober** (ketentuan proyek), lebih lebar daripada Jun–Okt pada `01_business_understanding.md` (asumsi A5). Seluruh fitur memakai definisi Mei–Okt.
2. **Kelompok lokasi** memakai nama "Jalan Raya" (sebelumnya "Tepi Jalan") agar konsisten dengan ketentuan proyek; `src/config.py` dan notebook 02 diperbarui.
3. **Tanggal harian memakai WIB** sesuai pencatatan sumber; `jam_lokal` (WITA/WIT) disediakan hanya untuk membaca pola harian. Tanggal kasus kesehatan diasumsikan sejajar dengan tanggal WIB (selisih ≤ 2 jam untuk stasiun WITA/WIT).
4. **ISPU hanya untuk PM2.5** (breakpoint dari CLAUDE.md); parameter lain belum dihitung karena breakpoint belum diverifikasi terhadap naskah Permen LHK 14/2020.
5. **Rata-rata 3 hari** (`pm25_ma3`) mensyaratkan 3 hari lengkap (tanpa `min_periods` longgar); lag dihitung per hari **kalender** pada grid lengkap sehingga hari tanpa data menjadi `NaN`, bukan menggeser baris.
6. **Status analisis** (Layak ≥ 75% pada jam, hari, dan hari kemarau; Terbatas; Dikecualikan < 10% jam) adalah ambang praktis proyek, bukan standar.
7. **Arah angin** dirata-rata secara melingkar; rata-rata harian arah angin juga melingkar.
8. Baris kesehatan 2023-12-31 (5 baris) dibuang karena tidak punya pasangan paparan.

## 7. Keterbatasan
- Hanya baris `VALID` yang dipakai; kebijakan ini memangkas puncak asap pada SM2-01 (dan mungkin sensor lain) sehingga estimasi dampak kabut asap bias ke bawah. Kolom `pm25_ug_m3_longgar` hanya untuk sensitivitas.
- Konversi PM "ppm" → µg/m³ (×1000) adalah hipotesis yang tervalidasi lewat orde besaran, bukan lewat pengukuran referensi.
- `pm_ekstrem` (PM > 500 µg/m³) hanya penanda untuk ditelaah; nilai ekstrem asli tidak dibuang, tetapi tidak dapat dibedakan secara pasti dari galat alat tanpa data pendukung (CO, hotspot).
- Stasiun BJB-01 dan SM2-01 memiliki PM2.5 tinggi namun kasus RS rendah; pembersihan tidak mengubah hal ini (lihat notebook 02), sehingga hasil hubungan PM2.5–kesehatan harus dilaporkan dengan sensitivitas tanpa stasiun tersebut.
- Tidak ada inferensi kausal pada tahap ini.

## 8. Cara menjalankan ulang
`python etl/run_preprocessing.py` membangun ulang seluruh keluaran (± 30 detik, deterministik). Skrip berhenti tanpa menulis apa pun bila satu validasi gagal.
